# JSBSim C172 Reference Simple Trim Validation v47

This notebook intentionally avoids the broad `fdm.do_trim(0)` grid that produced `TrimFailureError: Trim Failed` for every case.

The JSBSim C172 reference scripts and developer-list examples commonly use a simpler C172-specific flow:

1. Use `c172x`.
2. Initialize near cruise, usually with `reset01` or an airborne IC.
3. Start engine / set throttle, mixture, magnetos, starter.
4. Set flap/gear to clean airborne configuration.
5. Trigger `simulation/do_simple_trim = 1`.
6. Read the resulting elevator/throttle/state and validate with a fixed-control hold.

The goal here is not MPC/PID comparison. It is only to answer: can JSBSim itself produce a usable C172 trim-like starting state?

In [ ]:
# Colab only. If jsbsim is already installed, this is quick.
try:
    import jsbsim
except Exception:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'jsbsim', '-q'])
    import jsbsim
print('jsbsim import ok')

In [ ]:
import os, time, math, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import jsbsim

try:
    from google.colab import drive
    drive.mount('/content/drive')
    COLAB_DRIVE_MOUNTED = True
except Exception as exc:
    print('Drive mount skipped or unavailable:', exc)
    COLAB_DRIVE_MOUNTED = False

FPS_PER_KT = 1.687809857
EXPERIMENT_NAME = 'jsbsim_c172_reference_simple_trim_v47'
RUN_MODE = 'simple_trim_reference'
RESULT_ROOT = '/content/drive/MyDrive/Colab Result' if COLAB_DRIVE_MOUNTED else './Colab Result'
SAVE_DIR = os.path.join(RESULT_ROOT, 'PINN_MPC', EXPERIMENT_NAME)
os.makedirs(SAVE_DIR, exist_ok=True)
print('SAVE_DIR:', SAVE_DIR)

## 1. Configuration

The primary seed follows the JSBSim C172 cruise examples rather than the failed broad grid:

- aircraft: `c172x`
- initial condition: airborne cruise-like state
- speed: 90-100 kt sweep, with 90 kt included because the JSBSim mailing-list C172 trim example used 90 kt at 5000 ft
- altitude: 4000/5000/8000 ft
- throttle: 0.50 is included because `c172_cruise_8K.xml` starts engine with throttle command 0.50
- mixture: 1.0
- trim command: `simulation/do_simple_trim = 1`

In [ ]:
AIRCRAFT_GRID = ['c172x', 'c172p']
ALTITUDE_GRID_FT = [4000.0, 5000.0, 8000.0]
SPEED_GRID_KTS = [90.0, 95.0, 100.0, 105.0]
THROTTLE_GRID = [0.50, 0.65, 0.72, 0.85]
MIXTURE_GRID = [1.0, 0.87]
PITCH_GRID_DEG = [0.0, -1.0, 1.0]

DT = 0.02
SETTLE_SECONDS_BEFORE_TRIM = 1.0
HOLD_SECONDS = 60.0
TAIL_SECONDS = 10.0
TOP_K = 20

print('grid cases:', len(AIRCRAFT_GRID) * len(ALTITUDE_GRID_FT) * len(SPEED_GRID_KTS) * len(THROTTLE_GRID) * len(MIXTURE_GRID) * len(PITCH_GRID_DEG))

In [ ]:
def safe_set(fdm, prop, value):
    try:
        fdm[prop] = value
        return True
    except Exception:
        return False


def get_prop(fdm, prop, default=np.nan):
    try:
        return float(fdm[prop])
    except Exception:
        return float(default)


def read_state(fdm):
    return {
        'time_s': get_prop(fdm, 'simulation/sim-time-sec'),
        'h_sl_ft': get_prop(fdm, 'position/h-sl-ft'),
        'h_agl_ft': get_prop(fdm, 'position/h-agl-ft'),
        'vc_kts': get_prop(fdm, 'velocities/vc-kts'),
        'vt_fps': get_prop(fdm, 'velocities/vt-fps'),
        'u_fps': get_prop(fdm, 'velocities/u-fps'),
        'theta_deg': math.degrees(get_prop(fdm, 'attitude/theta-rad', 0.0)),
        'alpha_deg': math.degrees(get_prop(fdm, 'aero/alpha-rad', 0.0)),
        'q_deg_s': math.degrees(get_prop(fdm, 'velocities/q-rad_sec', 0.0)),
        'udot_fps2': get_prop(fdm, 'accelerations/udot-ft_sec2'),
        'wdot_fps2': get_prop(fdm, 'accelerations/wdot-ft_sec2'),
        'qdot_deg_s2': math.degrees(get_prop(fdm, 'accelerations/qdot-rad_sec2', 0.0)),
        'elevator_cmd': get_prop(fdm, 'fcs/elevator-cmd-norm'),
        'elevator_pos': get_prop(fdm, 'fcs/elevator-pos-norm'),
        'throttle_cmd': get_prop(fdm, 'fcs/throttle-cmd-norm'),
        'throttle_pos': get_prop(fdm, 'fcs/throttle-pos-norm[0]'),
        'mixture_cmd': get_prop(fdm, 'fcs/mixture-cmd-norm'),
        'rpm': get_prop(fdm, 'propulsion/engine/propeller-rpm'),
        'thrust_lbs': get_prop(fdm, 'propulsion/engine/thrust-lbs'),
    }


def make_fdm_airborne(aircraft, altitude_ft, speed_kts, pitch_deg, throttle, mixture):
    fdm = jsbsim.FGFDMExec(None, None)
    fdm.set_debug_level(0)
    fdm.load_model(aircraft)

    # Airborne IC aliases. Different JSBSim wheels expose slightly different aliases.
    for prop in ['ic/h-sl-ft', 'ic/altitude-ft']:
        safe_set(fdm, prop, altitude_ft)
    for prop in ['ic/vc-kts', 'ic/vt-kts']:
        safe_set(fdm, prop, speed_kts)
    safe_set(fdm, 'ic/u-fps', speed_kts * FPS_PER_KT)
    safe_set(fdm, 'ic/ubody-fps', speed_kts * FPS_PER_KT)
    safe_set(fdm, 'ic/v-fps', 0.0)
    safe_set(fdm, 'ic/w-fps', 0.0)
    safe_set(fdm, 'ic/p-rad_sec', 0.0)
    safe_set(fdm, 'ic/q-rad_sec', 0.0)
    safe_set(fdm, 'ic/r-rad_sec', 0.0)
    safe_set(fdm, 'ic/phi-deg', 0.0)
    safe_set(fdm, 'ic/theta-deg', pitch_deg)
    safe_set(fdm, 'ic/psi-true-deg', 200.0)
    safe_set(fdm, 'ic/lat-gc-deg', 47.0)
    safe_set(fdm, 'ic/long-gc-deg', -122.0)

    fdm.run_ic()

    # Engine and clean airborne configuration, matching JSBSim examples as closely as property API allows.
    for prop, val in [
        ('fcs/throttle-cmd-norm', throttle),
        ('fcs/throttle-pos-norm[0]', throttle),
        ('fcs/mixture-cmd-norm', mixture),
        ('fcs/mixture-pos-norm[0]', mixture),
        ('propulsion/magneto_cmd', 3),
        ('propulsion/starter_cmd', 1),
        ('fcs/flap-cmd-norm', 0.0),
        ('fcs/flap-pos-deg', 0.0),
        ('gear/gear-cmd-norm', 1.0),
        ('gear/gear-pos-norm', 1.0),
    ]:
        safe_set(fdm, prop, val)
    return fdm


def hold_existing_fdm(fdm, elevator_cmd, throttle_cmd, seconds=HOLD_SECONDS):
    rows = []
    n = int(seconds / DT)
    for _ in range(n):
        safe_set(fdm, 'fcs/elevator-cmd-norm', elevator_cmd)
        safe_set(fdm, 'fcs/throttle-cmd-norm', throttle_cmd)
        fdm.run()
        rows.append(read_state(fdm))
    df = pd.DataFrame(rows)
    if len(df) == 0:
        return {'hold_ok': False, 'score': np.inf}, df
    h0 = float(df['h_sl_ft'].iloc[0])
    tail = df.tail(max(1, int(TAIL_SECONDS / DT)))
    summary = {
        'hold_ok': True,
        'hold_h_change_ft': float(df['h_sl_ft'].iloc[-1] - h0),
        'hold_h_rmse_ft': float(np.sqrt(np.mean((df['h_sl_ft'] - h0) ** 2))),
        'hold_tail_vs_fps': float((tail['h_sl_ft'].iloc[-1] - tail['h_sl_ft'].iloc[0]) / max(float(tail['time_s'].iloc[-1] - tail['time_s'].iloc[0]), 1e-6)),
        'hold_speed_change_kts': float(df['vc_kts'].iloc[-1] - df['vc_kts'].iloc[0]),
        'hold_theta_std_deg': float(df['theta_deg'].std()),
        'hold_q_rms_deg_s': float(np.sqrt(np.mean(df['q_deg_s'] ** 2))),
        'hold_max_abs_qdot_deg_s2': float(np.max(np.abs(df['qdot_deg_s2']))),
    }
    summary['score'] = (
        1.0 * abs(summary['hold_h_change_ft'])
        + 2.0 * abs(summary['hold_tail_vs_fps'])
        + 1.5 * abs(summary['hold_speed_change_kts'])
        + 5.0 * summary['hold_q_rms_deg_s']
        + 0.5 * summary['hold_theta_std_deg']
    )
    return summary, df

## 2. Reference Simple Trim Grid

This uses only `simulation/do_simple_trim = 1`. It does **not** call `fdm.do_trim(0)`.

In [ ]:
def run_simple_trim_case(aircraft, altitude_ft, speed_kts, pitch_deg, throttle, mixture, store_log=False):
    row = {
        'aircraft': aircraft,
        'seed_altitude_ft': float(altitude_ft),
        'seed_speed_kts': float(speed_kts),
        'seed_pitch_deg': float(pitch_deg),
        'seed_throttle': float(throttle),
        'seed_mixture': float(mixture),
        'success': False,
        'status': 'not_run',
        'exception_type': '',
        'exception_message': '',
    }
    log_df = None
    try:
        fdm = make_fdm_airborne(aircraft, altitude_ft, speed_kts, pitch_deg, throttle, mixture)
        before = read_state(fdm)
        for _ in range(int(SETTLE_SECONDS_BEFORE_TRIM / DT)):
            fdm.run()
        pretrim = read_state(fdm)
        safe_set(fdm, 'simulation/do_simple_trim', 1)
        fdm.run()
        after = read_state(fdm)
        trim_elevator = after['elevator_cmd']
        trim_throttle = after['throttle_cmd'] if np.isfinite(after['throttle_cmd']) else throttle
        hold_summary, log_df = hold_existing_fdm(fdm, trim_elevator, trim_throttle, HOLD_SECONDS)
        row.update({f'before_{k}': v for k, v in before.items()})
        row.update({f'pretrim_{k}': v for k, v in pretrim.items()})
        row.update({f'trim_{k}': v for k, v in after.items()})
        row.update({
            'success': True,
            'status': 'simple_trim_triggered',
            'trim_elevator_cmd': float(trim_elevator),
            'trim_throttle_cmd': float(trim_throttle),
        })
        row.update(hold_summary)
    except Exception as exc:
        row['status'] = f'exception:{type(exc).__name__}'
        row['exception_type'] = type(exc).__name__
        row['exception_message'] = str(exc)
        row['score'] = np.inf
    return row, log_df

rows = []
best_log = None
best_score = np.inf
case_idx = 0
total = len(AIRCRAFT_GRID) * len(ALTITUDE_GRID_FT) * len(SPEED_GRID_KTS) * len(THROTTLE_GRID) * len(MIXTURE_GRID) * len(PITCH_GRID_DEG)
t0 = time.time()
for aircraft in AIRCRAFT_GRID:
    for altitude_ft in ALTITUDE_GRID_FT:
        for speed_kts in SPEED_GRID_KTS:
            for throttle in THROTTLE_GRID:
                for mixture in MIXTURE_GRID:
                    for pitch_deg in PITCH_GRID_DEG:
                        case_idx += 1
                        row, log_df = run_simple_trim_case(aircraft, altitude_ft, speed_kts, pitch_deg, throttle, mixture, store_log=False)
                        rows.append(row)
                        if np.isfinite(row.get('score', np.inf)) and row['score'] < best_score:
                            best_score = row['score']
                            _, best_log = run_simple_trim_case(aircraft, altitude_ft, speed_kts, pitch_deg, throttle, mixture, store_log=True)
                        if case_idx % 50 == 0:
                            tmp = pd.DataFrame(rows).sort_values('score')
                            best = tmp.iloc[0]
                            print(f'{case_idx}/{total} elapsed={time.time()-t0:.1f}s best score={best.score:.3f} aircraft={best.aircraft} V={best.seed_speed_kts:.1f} alt={best.seed_altitude_ft:.0f} thr={best.seed_throttle:.2f} mix={best.seed_mixture:.2f}')

simple_trim_df = pd.DataFrame(rows).sort_values('score').reset_index(drop=True)
print('Finished simple trim reference grid. Success count:', int(simple_trim_df['success'].sum()), '/', len(simple_trim_df))
display(simple_trim_df.head(TOP_K))

In [ ]:
csv_path = os.path.join(SAVE_DIR, f'c172_reference_simple_trim_v47_{RUN_MODE}.csv')
simple_trim_df.to_csv(csv_path, index=False)
print('Saved:', csv_path)

print('Success counts:')
display(simple_trim_df.groupby(['aircraft', 'success', 'exception_type'], dropna=False).size().reset_index(name='count'))

best = simple_trim_df.iloc[0]
print('Best candidate:')
for col in [
    'aircraft','seed_altitude_ft','seed_speed_kts','seed_pitch_deg','seed_throttle','seed_mixture',
    'trim_elevator_cmd','trim_throttle_cmd','trim_theta_deg','trim_vc_kts','trim_alpha_deg',
    'hold_h_change_ft','hold_h_rmse_ft','hold_tail_vs_fps','hold_speed_change_kts','hold_q_rms_deg_s','score'
]:
    if col in best:
        print(f'{col}: {best[col]}')

In [ ]:
if best_log is not None and len(best_log):
    fig, axes = plt.subplots(5, 1, figsize=(11, 11), sharex=True)
    axes[0].plot(best_log['time_s'], best_log['h_sl_ft']); axes[0].set_ylabel('Altitude ft')
    axes[1].plot(best_log['time_s'], best_log['vc_kts']); axes[1].set_ylabel('Vc kt')
    axes[2].plot(best_log['time_s'], best_log['theta_deg']); axes[2].set_ylabel('Theta deg')
    axes[3].plot(best_log['time_s'], best_log['q_deg_s']); axes[3].set_ylabel('q deg/s')
    axes[4].plot(best_log['time_s'], best_log['elevator_cmd'], label='elevator')
    axes[4].plot(best_log['time_s'], best_log['throttle_cmd'], label='throttle')
    axes[4].legend(); axes[4].set_ylabel('Controls'); axes[4].set_xlabel('Time s')
    for ax in axes:
        ax.grid(True, alpha=0.3)
    fig.suptitle('Best JSBSim C172 simple-trim fixed-control hold')
    plt.tight_layout()
    fig_path = os.path.join(SAVE_DIR, f'c172_reference_simple_trim_best_hold_v47_{RUN_MODE}.png')
    fig.savefig(fig_path, dpi=160)
    print('Saved:', fig_path)
    plt.show()
else:
    print('No best log available.')